Telecom Churn Dataset

Goal

The goal of this project is to predict customer churn for a telecom company using historical customer data. The model aims to identify customers who are likely to leave so that the business can take proactive retention actions.

Data

In [23]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report ,confusion_matrix
from sklearn.metrics import accuracy_score , precision_score,recall_score, f1_score, roc_auc_score


In [2]:
df = pd.read_csv('Telco_Churn_Dataset.csv')

In [85]:
df.info();df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   int64  
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   int64  
 3   Dependents        7032 non-null   int64  
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   int64  
 6   MultipleLines     7032 non-null   int64  
 7   InternetService   7032 non-null   int64  
 8   OnlineSecurity    7032 non-null   int64  
 9   OnlineBackup      7032 non-null   int64  
 10  DeviceProtection  7032 non-null   int64  
 11  TechSupport       7032 non-null   int64  
 12  StreamingTV       7032 non-null   int64  
 13  StreamingMovies   7032 non-null   int64  
 14  Contract          7032 non-null   int64  
 15  PaperlessBilling  7032 non-null   int64  
 16  PaymentMethod     7032 non-null   int64  
 17  

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


The dataset contains information about telecom customers, including demographic attributes, subscribed services, contract type, payment method, and monthly and total charges. The target variable Churn indicates whether the customer left the company in the last month.

EDA

In this section, we explore the distribution of key features and compare churn and non-churn groups. This helps to identify which customer segments and services are most strongly associated with churn.

In [4]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [5]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Preprocessing

During preprocessing, we clean the raw data, handle missing or inconsistent values, and convert categorical variables into numeric format suitable for modeling. Numerical features such as TotalCharges are properly typed and scaled, while the target variable Churn is encoded as a binary label.

In [6]:
df = df.drop(['customerID'],axis= 1)

In [7]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [8]:
df = df.dropna()

In [9]:
le = LabelEncoder()
for col in['gender','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod','Churn']:
    df[col]=le.fit_transform(df[col])

In [10]:
df.corr()['Churn'].sort_values(ascending=False)

Churn               1.000000
MonthlyCharges      0.192858
PaperlessBilling    0.191454
SeniorCitizen       0.150541
PaymentMethod       0.107852
MultipleLines       0.038043
PhoneService        0.011691
gender             -0.008545
StreamingTV        -0.036303
StreamingMovies    -0.038802
InternetService    -0.047097
Partner            -0.149982
Dependents         -0.163128
DeviceProtection   -0.177883
OnlineBackup       -0.195290
TotalCharges       -0.199484
TechSupport        -0.282232
OnlineSecurity     -0.289050
tenure             -0.354049
Contract           -0.396150
Name: Churn, dtype: float64

Modeling

We train several classification models, including Logistic Regression, Random Forest, and XGBoost, to predict customer churn. The models are evaluated on a hold-out test set using multiple metrics to capture both overall accuracy and performance on the churn class.

In [11]:
X =df.drop('Churn',axis= 1)
y =df['Churn']

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2 , random_state= 12)

In [13]:
model_1 = LogisticRegression(class_weight='balanced')
model_2 = RandomForestClassifier(n_estimators= 200, max_depth= 5,class_weight='balanced' )

In [50]:
model_3 = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=90,
    max_depth=2,
    learning_rate=0.08,
    subsample=0.7,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=12,
    scale_pos_weight = 2.769
)

In [15]:
model_1.fit(X_train,y_train)

c:\Users\nikit\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced')

In [16]:
y_pred_1 = model_1.predict(X_test)

In [17]:
print(confusion_matrix(y_test,y_pred_1))
print(classification_report(y_test,y_pred_1))

[[769 293]
 [ 75 270]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.81      1062
           1       0.48      0.78      0.59       345

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.81      0.74      0.75      1407



In [18]:
model_2.fit(X_train,y_train)

RandomForestClassifier(class_weight='balanced', max_depth=5, n_estimators=200)

In [19]:
y_pred_2 = model_2.predict(X_test)
print(confusion_matrix(y_test,y_pred_2))
print(classification_report(y_test,y_pred_2))

[[757 305]
 [ 74 271]]
              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1062
           1       0.47      0.79      0.59       345

    accuracy                           0.73      1407
   macro avg       0.69      0.75      0.69      1407
weighted avg       0.80      0.73      0.75      1407



In [51]:
model_3.fit(X_train,y_train)

c:\Users\nikit\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:22:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.08, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=90, n_jobs=None,
              num_parallel_tree=None, ...)

In [44]:
y_pred_3 = model_3.predict(X_test)
print(confusion_matrix(y_test,y_pred_3))
print(classification_report(y_test,y_pred_3))

[[770 292]
 [ 72 273]]
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1062
           1       0.48      0.79      0.60       345

    accuracy                           0.74      1407
   macro avg       0.70      0.76      0.70      1407
weighted avg       0.81      0.74      0.76      1407



Metrics / Statistics

For each model, we compute accuracy, precision, recall, F1-score for the churn class, and ROC AUC to measure the ability to distinguish churners from non-churners. These metrics are summarized in a single table, which makes it easy to compare model performance and choose the most suitable approach for this problem.

In [52]:
models = [
    ("LogReg",       model_1),
    ("RandomForest", model_2),
    ("XGBoost",      model_3),
]

rows = []

for name, model in models:
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # для roc_auc нужен score/вероятность[web:274]

    rows.append({
        "model":     name,
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall":    recall_score(y_test, y_pred, pos_label=1),
        "f1":        f1_score(y_test, y_pred, pos_label=1),
        "roc_auc":   roc_auc_score(y_test, y_proba),
    })

metrics_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).round(3)
metrics_df

,model,accuracy,precision,recall,f1,roc_auc
2,XGBoost,0.729,0.468,0.788,0.587,0.835
1,RandomForest,0.731,0.470,0.786,0.588,0.834
0,LogReg,0.738,0.480,0.783,0.595,0.825


Conclusion

Overall, the models achieve solid performance on the Telco churn task, with tree-based methods and XGBoost providing the best trade-off between recall and overall discrimination (ROC AUC). The analysis highlights that contract type, tenure, additional services, and payment behavior are among the most important drivers of churn, and the final model can be used as a foundation for targeted retention strategies.